# 13 · From imitation to preference

**Recap (01–12):** every PEFT method so far changed the **mechanism** axis — *how
many* parameters we train (LoRA, QLoRA, BitFit, IA3, prompt, prefix). They all used
the **same objective**: copy the gold answer.

Now we switch axes. The **objective** axis changes *what* we train on. This notebook
sets it up: what our old objective (SFT) really optimizes, and why we'd want a new
one. New primitive: the **log-probability** of a sequence.

In [1]:
import numpy as np
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

## Step 1 — A model puts a *probability* on each next token

At each step a language model outputs a score for every possible next token, and
**softmax** turns those scores into probabilities that sum to 1. SFT (supervised
fine-tuning, our objective so far) simply **pushes up the probability of the gold
token** at each step.

In [2]:
vocab  = ["SELECT", "FROM", "COUNT", "customers", "orders"]
logits = np.array([2.0, 1.0, 0.5, 1.5, 0.2])    # the model's raw scores for the next token
probs  = softmax(logits)

gold = vocab.index("SELECT")
print("next-token probabilities:", dict(zip(vocab, probs.round(3))))
print("prob of gold 'SELECT'   :", probs[gold].round(3))
print("log-prob of gold        :", np.log(probs[gold]).round(3))

next-token probabilities: {'SELECT': 0.423, 'FROM': 0.156, 'COUNT': 0.094, 'customers': 0.257, 'orders': 0.07}
prob of gold 'SELECT'   : 0.423
log-prob of gold        : -0.86


## Step 2 — A whole answer's log-probability = a *sum* of log-probs

A SQL answer is many tokens. The probability of the whole sequence is the product of
the per-token probabilities — but products of many small numbers underflow, so we
add **log-probabilities** instead (log turns products into sums). Higher sequence
log-prob = the model finds that answer more likely.

In [3]:
per_token_prob = np.array([0.6, 0.4, 0.7])      # gold-token probability at each step

print("sequence prob (product) :", per_token_prob.prod().round(3))
print("sequence log-prob (sum) :", np.log(per_token_prob).sum().round(3))
print("SFT raises this gold log-prob; that's the entire SFT objective.")

sequence prob (product) : 0.168
sequence log-prob (sum) : -1.784
SFT raises this gold log-prob; that's the entire SFT objective.


## Step 3 — Why imitation alone is brittle

SFT only ever says *"this exact gold answer is correct."* It never learns that some
**wrong** answers are worse than others, and it can't say *"I prefer answer A over
answer B."* For text-to-SQL many different queries are acceptable, and pure imitation
of one gold string is fragile.

The fix: stop handing the model a single gold answer. Instead hand it **pairs** — a
better answer (**chosen**) and a worse one (**rejected**) — and train it to make the
chosen one more probable than the rejected one. That's **preference tuning**, and
**DPO** is the simplest way to do it.

## Recap

- A model assigns a **probability** to each next token (via softmax); a sequence's
  **log-probability** is the **sum** of per-token log-probs.
- **SFT** just raises the gold sequence's log-prob — pure imitation.
- Imitation can't express *preferences*; preference tuning trains on **(chosen,
  rejected)** pairs instead.

**BAM!** Next: **`14 — where do the pairs come from?`**